In [6]:
import zipfile
import numpy as np
from PIL import Image
from io import BytesIO

X = []
y = []

with zipfile.ZipFile("archive.zip", "r") as z:
    for file in z.namelist():

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            parts = file.split("/")

            # Adjust index according to your folder structure
            emotion = parts[-2]

            img = Image.open(BytesIO(z.read(file))).convert("L")
            img = np.array(img, dtype=np.float32) / 255.0

            X.append(img)
            y.append(emotion)

X = np.array(X)
y = np.array(y)

print("Images shape:", X.shape)
print("Labels shape:", y.shape)

Images shape: (35215, 48, 48)
Labels shape: (35215,)


In [11]:
import zipfile
import numpy as np
from PIL import Image
from io import BytesIO
from sklearn.model_selection import train_test_split

X = []
y = []

with zipfile.ZipFile("archive.zip", "r") as z:

    for file in z.namelist():

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            parts = file.split("/")

            # emotion folder name
            emotion = parts[-2]

            img = Image.open(BytesIO(z.read(file))).convert("L")

            img = np.array(img, dtype=np.float32) / 255.0

            X.append(img)
            y.append(emotion)

X = np.array(X)
y = np.array(y)

# 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split temp into validation and test (10% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (28172, 48, 48)
Validation: (3521, 48, 48)
Test: (3522, 48, 48)


In [14]:
from collections import Counter
from sklearn.utils import resample
import numpy as np

# Count samples per class
class_counts = Counter(y_train)
max_count = max(class_counts.values())

X_balanced = []
y_balanced = []

for cls in np.unique(y_train):

    X_cls = X_train[y_train == cls]
    y_cls = y_train[y_train == cls]

    X_resampled, y_resampled = resample(
        X_cls,
        y_cls,
        replace=True,
        n_samples=max_count,
        random_state=42
    )

    X_balanced.append(X_resampled)
    y_balanced.append(y_resampled)

X_train_balanced = np.vstack(X_balanced)
y_train_balanced = np.hstack(y_balanced)

print(Counter(y_train_balanced))

Counter({np.str_('angry'): 7002, np.str_('disgust'): 7002, np.str_('fear'): 7002, np.str_('happy'): 7002, np.str_('neutral'): 7002, np.str_('sad'): 7002, np.str_('surprise'): 7002})


In [16]:
print(type(y_train[0]))
print(y_train[:10])

<class 'numpy.str_'>
['happy' 'surprise' 'fear' 'angry' 'angry' 'happy' 'sad' 'happy' 'sad'
 'happy']
